# Bronze: Latest Measurements Ingestion

**Purpose**: Fetch latest air quality measurements from OpenAQ API for active Canadian monitoring stations\
**Schedule**: Every hour\
**Source**: OpenAQ API v3 `/locations/{id}/latest` endpoint\
**Target**: `airquality.bronze.measurements`

### 1.0: Configuration

In [0]:
%run ./00_utils

### 2.0: API Call Functions

In [ ]:
import time

def fetch_all_latest_measurements(location_ids):
    """
    Call /locations/{id}/latest once per station and collect all readings.

    A failing station is logged and skipped, so one bad API response never
    kills the whole hourly run. Sleeps 1 second between calls to respect
    the API rate limit (60 requests/min).
    """
    latest_measurements = []

    for i, loc_id in enumerate(location_ids):
        try:
            data = fetch_openaq(f"locations/{loc_id}/latest")
            latest_measurements.extend(data.get("results", []))
        except Exception as e:
            print(f"Error fetching location {loc_id}: {e}")

        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(location_ids)} locations")

        time.sleep(1)

    return latest_measurements

### 3.0: Run ingestion

In [ ]:
from datetime import datetime, timedelta, timezone

# Poll only stations that reported in the last 24 hours: at 60 calls/min,
# skipping dead stations keeps the hourly job fast.
cutoff = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Cutoff: {cutoff}")

# Bronze holds one snapshot per location per day; DISTINCT collapses the
# repeated ids to one row each.
df_ids = spark.sql(f"""
    SELECT DISTINCT id
    FROM {CATALOG}.{SCHEMA}.locations
    WHERE get_json_object(datetimeLast, '$.utc') >= '{cutoff}'
""")
location_ids = [row.id for row in df_ids.collect()]

print(f"Fetching latest measurements for {len(location_ids)} active locations...")
print(f"Estimated time: ~{len(location_ids) // 60 + 1} minutes (rate limit: 60 calls/min)")

measurements = fetch_all_latest_measurements(location_ids)

print(f"\nTotal measurements fetched: {len(measurements)}")

### 4.0: Clean Data

In [ ]:
import json

# Keep bronze close to the raw API payload: scalars become columns, nested
# objects are stored as JSON strings. Parsing happens in the silver notebook.
measurements_clean = []
for m in measurements:
    clean = {
        "sensors_id": m.get("sensorsId"),
        "locations_id": m.get("locationsId"),
        "value": m.get("value"),
        "datetime": json.dumps(m.get("datetime")),
        "coordinates": json.dumps(m.get("coordinates")),
    }
    measurements_clean.append(clean)

print(f"Prepared {len(measurements_clean)} measurements for bronze")

### 5.0: Save Measurements to Bronze

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

# Convert to Spark and stamp every row with load metadata:
# ingested_at powers the incremental watermark in the silver notebook.
df = spark.createDataFrame(pd.DataFrame(measurements_clean))

df_bronze = df.withColumn("ingested_at", F.lit(datetime.now(timezone.utc).isoformat())) \
              .withColumn("source", F.lit("openaq_api_v3_latest"))

# Append only: bronze is the immutable history we can always replay from.
# Duplicates across hourly runs are expected here; silver removes them.
df_bronze.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.measurements")

print(f"Appended {len(measurements_clean)} measurements to {CATALOG}.{SCHEMA}.measurements")